# EDA for MOEX options data

Этот ноутбук проверяет файлы, которые были скачаны в `data/raw`, и отвечает на три вопроса:
1. Похожи ли данные на корректные MOEX ISS выгрузки.
2. Насколько они полные и качественные.
3. Достаточны ли они для дальнейшего анализа.

Предварительный вывод по текущему набору:
- файл свечей IMOEX выглядит корректно и качественно;
- файл кандидатов по опционам выглядит правдоподобно по типам и названиям;
- файл истории опционов сейчас очень слабый: в нём только 1 SECID, только 2024 год и почти все рыночные поля пустые;
- значит, discovery и/или выбор исторических SECID требует доработки перед серьёзным анализом.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path('/Users/maria/Desktop/Code/HSE/COURSEBOOK')
RAW_DIR = ROOT / 'data' / 'raw'

metadata_path = RAW_DIR / 'moex_options_2y_metadata_raw.parquet'
candidates_path = RAW_DIR / 'moex_options_2y_candidates.parquet'
history_path = RAW_DIR / 'moex_options_2y_history.parquet'
candles_path = RAW_DIR / 'moex_imoex_2y_candles.parquet'
report_path = RAW_DIR / 'moex_options_2y_report.txt'

for path in [metadata_path, candidates_path, history_path, candles_path, report_path]:
    print(path.name, 'exists =' , path.exists())

moex_options_2y_metadata_raw.parquet exists = True
moex_options_2y_candidates.parquet exists = True
moex_options_2y_history.parquet exists = True
moex_imoex_2y_candles.parquet exists = True
moex_options_2y_report.txt exists = True


In [2]:
meta = pd.read_parquet(metadata_path)
cand = pd.read_parquet(candidates_path)
hist = pd.read_parquet(history_path)
imoex = pd.read_parquet(candles_path)

if 'TRADEDATE' in hist.columns:
    hist['TRADEDATE'] = pd.to_datetime(hist['TRADEDATE'], errors='coerce')
if 'begin' in imoex.columns:
    imoex['begin'] = pd.to_datetime(imoex['begin'], errors='coerce')
if 'end' in imoex.columns:
    imoex['end'] = pd.to_datetime(imoex['end'], errors='coerce')

datasets = {
    'metadata_raw': meta,
    'candidates': cand,
    'history': hist,
    'imoex_candles': imoex,
}

pd.DataFrame({
    name: {
        'rows': len(df),
        'cols': df.shape[1],
        'columns': ', '.join(df.columns[:8]) + (' ...' if df.shape[1] > 8 else ''),
    }
    for name, df in datasets.items()
}).T

,rows,cols,columns
metadata_raw,246,15,"secid, shortname, regnumber, name, isin, is_tr..."
candidates,160,17,"secid, shortname, regnumber, name, isin, is_tr..."
history,170,16,"BOARDID, TRADEDATE, SECID, OPEN, LOW, HIGH, CL..."
imoex_candles,500,8,"open, close, high, low, value, volume, begin, end"


## 1. Сырые метаданные discovery

Проверяем, не затянул ли discovery слишком много нерелевантных инструментов.

In [3]:
display(meta.head(10))

print('Rows:', len(meta))
print('Unique SECIDs:', meta['secid'].nunique() if 'secid' in meta.columns else 'n/a')
print('\nType distribution:')
display(meta['type'].value_counts(dropna=False).to_frame('count').head(20))
print('\nGroup distribution:')
display(meta['group'].value_counts(dropna=False).to_frame('count').head(20))
print('\nMissing share by column (%):')
display((meta.isna().mean().sort_values(ascending=False) * 100).round(1).to_frame('missing_pct'))

,secid,shortname,regnumber,name,isin,is_traded,emitent_id,emitent_title,emitent_inn,emitent_okpo,type,group,primary_boardid,marketprice_boardid,_query
0,IMOEX,Индекс МосБиржи,None,Индекс МосБиржи,RU000A0JP7K5,1,NaN,None,None,None,stock_index,stock_index,SNDX,None,IMOEX
1,IMOEX2,Индекс МосБиржи (все сессии),None,IMOEX2 – значения индекса МосБиржи за весь тор...,None,1,NaN,None,None,None,stock_index_eq,stock_index,SNDX,None,IMOEX
2,IMOEXW,IMOEXW,None,Индекс МосБиржи – активное управление,None,1,NaN,None,None,None,stock_index_eq,stock_index,RTSI,None,IMOEX
3,IMOEXCNY,Индекс МосБиржи в юанях,None,Индекс МосБиржи в юанях,None,1,NaN,None,None,None,stock_index_eq,stock_index,RTSI,None,IMOEX
4,EQMXE,iNAV EQMX ETF,None,Расчетная цена одного пая биржевого ПИФа «Инде...,None,1,NaN,None,None,None,stock_index_ie,stock_index,INAV,None,IMOEX
5,IMOEXDIV,Индекс дивидендов МосБиржи,None,Индекс дивидендов МосБиржи «брутто»,None,1,NaN,None,None,None,stock_index_eq,stock_index,RTSI,None,IMOEX
6,IMOEXDIVN,Индекс див МосБиржи нетто-рез,None,Индекс дивидендов МосБиржи «нетто» (по налогов...,None,1,NaN,None,None,None,stock_index_eq,stock_index,RTSI,None,IMOEX
7,TMOSA,TMOS iNAV,None,Расчетная цена одного пая Биржевого ПИФа РФИ «...,None,1,NaN,None,None,None,stock_index,stock_index,INAV,None,IMOEX
8,RU000A106SD7,СберИОС582,4B02-612-01481-B-001P,СберИОС 001Р-582R 5Г AC IMOEX,RU000A106SD7,1,484.0,"Публичное акционерное общество ""Сбербанк России""",7707083893,00032537,exchange_bond,stock_bonds,TQCB,TQCB,IMOEX
9,RU000A103TL5,СберИОС409,4B02-415-01481-B-001P,СберИОС 001Р-409R 5Г IMOEX,RU000A103TL5,1,484.0,"Публичное акционерное общество ""Сбербанк России""",7707083893,00032537,exchange_bond,stock_bonds,TQCB,TQCB,IMOEX


Rows: 246
Unique SECIDs: 246

Type distribution:


,count
type,
option,104
option_on_indices,56
exchange_bond,41
futures,17
stock_index,8
stock_index_mx,8
stock_index_eq,5
futures_spread,3
stock_index_ie,1



Group distribution:


,count
group,
futures_options,160
stock_bonds,41
stock_index,22
futures_forts,21
stock_ppif,2



Missing share by column (%):


,missing_pct
regnumber,82.5
emitent_id,82.5
emitent_title,82.5
emitent_inn,82.5
emitent_okpo,82.5
marketprice_boardid,82.5
isin,82.1
secid,0.0
shortname,0.0
name,0.0


### Комментарий

Сырые метаданные содержат и нужные опционы, и много побочного мусора: облигации, индексы, фьючерсы и другие инструменты. Это нормально для узкого поиска по `q=IMOEX`, `q=MIX`, `q=MX`, но значит, что качество результата полностью зависит от следующего фильтра кандидатов.

## 2. Отфильтрованные кандидаты

Проверяем, действительно ли после фильтра остались именно опционы на фьючерсы, связанные с IMOEX.

In [4]:
display(cand.head(10))

print('Rows:', len(cand))
print('Unique SECIDs:', cand['secid'].nunique())
print('\nType distribution:')
display(cand['type'].value_counts(dropna=False).to_frame('count'))
print('\nGroup distribution:')
display(cand['group'].value_counts(dropna=False).to_frame('count'))
print('\nPrimary boards:')
display(cand['primary_boardid'].value_counts(dropna=False).to_frame('count').head(10))
print('\nCandidate reason distribution:')
display(cand['_candidate_reason'].value_counts(dropna=False).to_frame('count').head(10))

,secid,shortname,regnumber,name,isin,is_traded,emitent_id,emitent_title,emitent_inn,emitent_okpo,type,group,primary_boardid,marketprice_boardid,_query,_keep_candidate,_candidate_reason
0,IM2600CE6,IMOEXP200526CE2600,None,Прем. европ. Call 2600 с исп. 20 мая на IMOEX,None,1,None,None,None,None,option_on_indices,futures_options,ROPD,None,IMOEX,True,"matched_text,option_like"
1,IM2650CE6,IMOEXP200526CE2650,None,Прем. европ. Call 2650 с исп. 20 мая на IMOEX,None,1,None,None,None,None,option_on_indices,futures_options,ROPD,None,IMOEX,True,"matched_text,option_like"
2,IM2700CE6,IMOEXP200526CE2700,None,Прем. европ. Call 2700 с исп. 20 мая на IMOEX,None,1,None,None,None,None,option_on_indices,futures_options,ROPD,None,IMOEX,True,"matched_text,option_like"
3,IM2600CE6A,IMOEXP060526CE2600,None,Нед. прем. европ. Call 2600 с исп. 6 мая на IMOEX,None,0,None,None,None,None,option_on_indices,futures_options,ROPD,None,IMOEX,True,"matched_text,option_like"
4,IM2850CQ6A,IMOEXP060526PE2850,None,Нед. прем. европ. Put 2850 с исп. 6 мая на IMOEX,None,0,None,None,None,None,option_on_indices,futures_options,ROPD,None,IMOEX,True,"matched_text,option_like"
5,IM2650CQ6,IMOEXP200526PE2650,None,Прем. европ. Put 2650 с исп. 20 мая на IMOEX,None,1,None,None,None,None,option_on_indices,futures_options,ROPD,None,IMOEX,True,"matched_text,option_like"
6,IM2750CE6,IMOEXP200526CE2750,None,Прем. европ. Call 2750 с исп. 20 мая на IMOEX,None,1,None,None,None,None,option_on_indices,futures_options,ROPD,None,IMOEX,True,"matched_text,option_like"
7,IM2800CE6,IMOEXP200526CE2800,None,Прем. европ. Call 2800 с исп. 20 мая на IMOEX,None,1,None,None,None,None,option_on_indices,futures_options,ROPD,None,IMOEX,True,"matched_text,option_like"
8,IM2850CQ6,IMOEXP200526PE2850,None,Прем. европ. Put 2850 с исп. 20 мая на IMOEX,None,1,None,None,None,None,option_on_indices,futures_options,ROPD,None,IMOEX,True,"matched_text,option_like"
9,IM2750CD6D,IMOEXP220426CE2750,None,Нед. прем. европ. Call 2750 с исп. 22 апр. на ...,None,0,None,None,None,None,option_on_indices,futures_options,ROPD,None,IMOEX,True,"matched_text,option_like"


Rows: 160
Unique SECIDs: 160

Type distribution:


,count
type,
option,104
option_on_indices,56



Group distribution:


,count
group,
futures_options,160



Primary boards:


,count
primary_boardid,
ROPD,160



Candidate reason distribution:


,count
_candidate_reason,
"matched_text,option_like",160


In [5]:
secid_sample = cand['secid'].head(20).tolist()
name_sample = cand[['secid', 'shortname', 'name']].head(20)
print('First 20 candidate SECIDs:')
print(secid_sample)
display(name_sample)

First 20 candidate SECIDs:
['IM2600CE6', 'IM2650CE6', 'IM2700CE6', 'IM2600CE6A', 'IM2850CQ6A', 'IM2650CQ6', 'IM2750CE6', 'IM2800CE6', 'IM2850CQ6', 'IM2750CD6D', 'IM2600CD6E', 'IM2650CE6A', 'IM2500CE6', 'IM2650CD6D', 'IM2750CQ6', 'IM2800CQ6A', 'IM2500CE6A', 'IM2700CD6E', 'IM2700CQ6', 'IM2850CP6D']


,secid,shortname,name
0,IM2600CE6,IMOEXP200526CE2600,Прем. европ. Call 2600 с исп. 20 мая на IMOEX
1,IM2650CE6,IMOEXP200526CE2650,Прем. европ. Call 2650 с исп. 20 мая на IMOEX
2,IM2700CE6,IMOEXP200526CE2700,Прем. европ. Call 2700 с исп. 20 мая на IMOEX
3,IM2600CE6A,IMOEXP060526CE2600,Нед. прем. европ. Call 2600 с исп. 6 мая на IMOEX
4,IM2850CQ6A,IMOEXP060526PE2850,Нед. прем. европ. Put 2850 с исп. 6 мая на IMOEX
5,IM2650CQ6,IMOEXP200526PE2650,Прем. европ. Put 2650 с исп. 20 мая на IMOEX
6,IM2750CE6,IMOEXP200526CE2750,Прем. европ. Call 2750 с исп. 20 мая на IMOEX
7,IM2800CE6,IMOEXP200526CE2800,Прем. европ. Call 2800 с исп. 20 мая на IMOEX
8,IM2850CQ6,IMOEXP200526PE2850,Прем. европ. Put 2850 с исп. 20 мая на IMOEX
9,IM2750CD6D,IMOEXP220426CE2750,Нед. прем. европ. Call 2750 с исп. 22 апр. на ...


### Комментарий

Здесь фильтр выглядит разумно: остаются только `group = futures_options`, а типы состоят из `option` и `option_on_indices`. По названиям тоже видно, что это действительно опционы на IMOEX. То есть основная проблема, скорее всего, не в самих кандидатах как таковых, а в том, какие из них реально дают историю за нужный период.

## 3. История по опционам

Это главный блок проверки: хватает ли истории, правильный ли у неё формат, есть ли покрытие по годам и по SECID.

In [6]:
display(hist.head(10))

print('Rows:', len(hist))
print('Columns:', list(hist.columns))
print('Unique SECIDs in history:', hist['SECID'].nunique() if 'SECID' in hist.columns else 'n/a')
print('Date range:', hist['TRADEDATE'].min(), '->', hist['TRADEDATE'].max())
print('\nRows by year:')
display(hist['TRADEDATE'].dt.year.value_counts(dropna=False).sort_index().to_frame('count'))
print('\nRows by SECID (top 20):')
display(hist['SECID'].value_counts().head(20).to_frame('count'))

,BOARDID,TRADEDATE,SECID,OPEN,LOW,HIGH,CLOSE,OPENPOSITIONVALUE,VALUE,VOLUME,OPENPOSITION,SETTLEPRICE,WAPRICE,CHANGE,QTY,NUMTRADES
0,ROPD,2024-05-06,IM2750CF6,None,None,None,None,0.0,None,None,None,837.95,None,None,None,0
1,ROPD,2024-05-07,IM2750CF6,None,None,None,None,0.0,None,None,None,842.18,None,None,None,0
2,ROPD,2024-05-08,IM2750CF6,None,None,None,None,0.0,None,None,None,840.38,None,None,None,0
3,ROPD,2024-05-10,IM2750CF6,None,None,None,None,0.0,None,None,None,839.53,None,None,None,0
4,ROPD,2024-05-13,IM2750CF6,None,None,None,None,0.0,None,None,None,851.52,None,None,None,0
5,ROPD,2024-05-14,IM2750CF6,None,None,None,None,0.0,None,None,None,918.10,None,None,None,0
6,ROPD,2024-05-15,IM2750CF6,None,None,None,None,0.0,None,None,None,876.49,None,None,None,0
7,ROPD,2024-05-16,IM2750CF6,None,None,None,None,0.0,None,None,None,895.40,None,None,None,0
8,ROPD,2024-05-17,IM2750CF6,None,None,None,None,0.0,None,None,None,891.55,None,None,None,0
9,ROPD,2024-05-20,IM2750CF6,None,None,None,None,0.0,None,None,None,916.21,None,None,None,0


Rows: 170
Columns: ['BOARDID', 'TRADEDATE', 'SECID', 'OPEN', 'LOW', 'HIGH', 'CLOSE', 'OPENPOSITIONVALUE', 'VALUE', 'VOLUME', 'OPENPOSITION', 'SETTLEPRICE', 'WAPRICE', 'CHANGE', 'QTY', 'NUMTRADES']
Unique SECIDs in history: 1
Date range: 2024-05-06 00:00:00 -> 2024-12-30 00:00:00

Rows by year:


,count
TRADEDATE,
2024,170



Rows by SECID (top 20):


,count
SECID,
IM2750CF6,170


In [7]:
print('Missing share by column (%):')
display((hist.isna().mean().sort_values(ascending=False) * 100).round(1).to_frame('missing_pct'))

numeric_cols = [
    'OPEN', 'LOW', 'HIGH', 'CLOSE', 'OPENPOSITIONVALUE', 'VALUE', 'VOLUME',
    'OPENPOSITION', 'SETTLEPRICE', 'WAPRICE', 'CHANGE', 'QTY', 'NUMTRADES'
]
non_null_counts = {}
for col in numeric_cols:
    if col in hist.columns:
        non_null_counts[col] = pd.to_numeric(hist[col], errors='coerce').notna().sum()

display(pd.Series(non_null_counts, name='non_null_count').sort_values().to_frame())

Missing share by column (%):


,missing_pct
OPEN,100.0
LOW,100.0
HIGH,100.0
CLOSE,100.0
VALUE,100.0
VOLUME,100.0
OPENPOSITION,100.0
WAPRICE,100.0
CHANGE,100.0
QTY,100.0


,non_null_count
OPEN,0
LOW,0
HIGH,0
CLOSE,0
VALUE,0
VOLUME,0
OPENPOSITION,0
WAPRICE,0
CHANGE,0
QTY,0


In [8]:
candidate_secids = cand['secid'].head(50).astype(str).tolist()
history_secids = set(hist['SECID'].astype(str).unique()) if 'SECID' in hist.columns else set()
matched = sorted(set(candidate_secids) & history_secids)
missing = sorted(set(candidate_secids) - history_secids)

coverage = pd.DataFrame({
    'metric': ['candidate_secids_first_50', 'secids_with_history', 'secids_without_history'],
    'value': [len(candidate_secids), len(matched), len(missing)],
})
display(coverage)

print('Matched SECIDs with history:')
print(matched)
print('\nFirst SECIDs without history:')
print(missing[:30])

,metric,value
0,candidate_secids_first_50,50
1,secids_with_history,1
2,secids_without_history,49


Matched SECIDs with history:
['IM2750CF6']

First SECIDs without history:
['IM2500CE6', 'IM2500CE6A', 'IM2500CQ6', 'IM2550CE6', 'IM2550CE6A', 'IM2550CQ6', 'IM2600CD6D', 'IM2600CD6E', 'IM2600CE6', 'IM2600CE6A', 'IM2600CE6D', 'IM2600CQ6', 'IM2650CD6', 'IM2650CD6D', 'IM2650CD6E', 'IM2650CE6', 'IM2650CE6A', 'IM2650CQ6', 'IM2700CD6', 'IM2700CD6D', 'IM2700CD6E', 'IM2700CE6', 'IM2700CE6A', 'IM2700CQ6', 'IM2700CQ6A', 'IM2750CD6D', 'IM2750CD6E', 'IM2750CE6', 'IM2750CP6', 'IM2750CQ6']


### Ключевой вывод по истории опционов

Текущий файл истории выглядит **структурно корректным**, но **аналитически очень слабым**:
- схема колонок соответствует ожидаемой MOEX history-таблице;
- даты парсятся корректно;
- но покрытие почти отсутствует: история есть только у **1 SECID из 50**;
- весь датасет лежит только в **2024 году**, хотя целевой период был `2023-01-01` -- `2024-12-31`;
- почти все торговые поля (`OPEN`, `HIGH`, `LOW`, `CLOSE`, `VALUE`, `VOLUME`, `OPENPOSITION`, `WAPRICE`, `CHANGE`, `QTY`) пустые на 100%;
- фактически полезные данные есть только в `SETTLEPRICE`, `OPENPOSITIONVALUE` и `NUMTRADES`.

То есть это не похоже на полноценный двухлетний market history для набора IMOEX-опционов. Для дальнейшей работы нужно улучшать discovery исторических контрактов и проверять expired series.

## 4. Дневные свечи IMOEX

Проверяем референсный рыночный ряд по базовому индексу.

In [9]:
display(imoex.head(10))

print('Rows:', len(imoex))
print('Columns:', list(imoex.columns))
print('Date range:', imoex['begin'].min(), '->', imoex['begin'].max())
print('\nRows by year:')
display(imoex['begin'].dt.year.value_counts().sort_index().to_frame('count'))
print('\nMissing share by column (%):')
display((imoex.isna().mean().sort_values(ascending=False) * 100).round(1).to_frame('missing_pct'))
print('\nDuplicate begin timestamps:', imoex['begin'].duplicated().sum())

,open,close,high,low,value,volume,begin,end
0,2157.18,2172.68,2174.23,2157.18,1.269031e+10,0,2023-01-03,2023-01-03 23:59:59
1,2171.54,2168.42,2179.56,2162.06,1.151004e+10,0,2023-01-04,2023-01-04 23:59:59
2,2170.40,2156.67,2171.94,2154.16,9.783315e+09,0,2023-01-05,2023-01-05 23:59:59
3,2157.32,2156.39,2160.08,2153.32,7.622558e+09,0,2023-01-06,2023-01-06 23:59:59
4,2163.43,2163.50,2169.71,2162.01,1.817911e+10,0,2023-01-09,2023-01-09 23:59:59
5,2162.70,2159.51,2162.92,2145.14,1.460017e+10,0,2023-01-10,2023-01-10 23:59:59
6,2156.28,2186.98,2190.40,2153.55,3.525735e+10,0,2023-01-11,2023-01-11 23:59:59
7,2193.53,2185.93,2194.80,2177.36,2.145680e+10,0,2023-01-12,2023-01-12 23:59:59
8,2188.60,2199.94,2204.18,2179.82,2.768824e+10,0,2023-01-13,2023-01-13 23:59:59
9,2204.64,2224.90,2224.90,2204.46,2.733352e+10,0,2023-01-16,2023-01-16 23:59:59


Rows: 500
Columns: ['open', 'close', 'high', 'low', 'value', 'volume', 'begin', 'end']
Date range: 2023-01-03 00:00:00 -> 2024-12-17 00:00:00

Rows by year:


,count
begin,
2023,254
2024,246



Missing share by column (%):


,missing_pct
open,0.0
close,0.0
high,0.0
low,0.0
value,0.0
volume,0.0
begin,0.0
end,0.0



Duplicate begin timestamps: 0


### Комментарий

Свечи IMOEX выглядят хорошо:
- диапазон дат покрывает 2023 и 2024 годы;
- пропусков нет;
- дублей по дате нет;
- формат колонок выглядит стандартным для ISS candles endpoint.

Это хороший базовый ряд, его можно использовать как benchmark / underlying time series.

## 5. Сравнение с отчётом загрузчика

Полезно сразу видеть, что написано в текстовом отчёте скрипта.

In [10]:
print(report_path.read_text(encoding='utf-8'))

MOEX options 2y MVP report

tested discovery URLs
- https://iss.moex.com/iss/securities.json?iss.meta=off&q=IMOEX&limit=100
- https://iss.moex.com/iss/securities.json?iss.meta=off&q=MIX&limit=100
- https://iss.moex.com/iss/securities.json?iss.meta=off&q=MX&limit=100

metadata columns: secid, shortname, regnumber, name, isin, is_traded, emitent_id, emitent_title, emitent_inn, emitent_okpo, type, group, primary_boardid, marketprice_boardid, _query
number of raw metadata rows: 246
number of filtered candidates: 160
first 50 candidate SECIDs: IM2600CE6, IM2650CE6, IM2700CE6, IM2600CE6A, IM2850CQ6A, IM2650CQ6, IM2750CE6, IM2800CE6, IM2850CQ6, IM2750CD6D, IM2600CD6E, IM2650CE6A, IM2500CE6, IM2650CD6D, IM2750CQ6, IM2800CQ6A, IM2500CE6A, IM2700CD6E, IM2700CQ6, IM2850CP6D, IM2850CP6E, IM2900CQ6, IM2900CQ6A, IM2550CE6, IM2700CD6D, IM2700CE6A, IM2750CD6E, IM2850CE6, IM2600CQ6, IM2650CD6E, IM2700CQ6A, IM2800CQ6, IM2900CP6E, IM2600CD6D, IM2900CP6D, IM2550CE6A, IM2650CD6, IM2700CD6, IM2750CF6, IM275

## Итог

Итоговая оценка качества текущих файлов:

- `moex_options_2y_metadata_raw.parquet`: нормальный сырой discovery, но шумный.
- `moex_options_2y_candidates.parquet`: правдоподобный список кандидатов, выглядит разумно.
- `moex_options_2y_history.parquet`: формат корректный, но качество низкое для исследования, потому что покрытие почти нулевое.
- `moex_imoex_2y_candles.parquet`: качественный и пригодный ряд.

Главная проблема набора сейчас не в формате parquet и не в чтении данных, а в **содержательном покрытии option history**. Перед эконометрическим или финансовым анализом нужно заново улучшить discovery и добиться того, чтобы история возвращалась не для 1 SECID, а для заметной доли релевантных контрактов.